# 03 — Análise e Correlação

Análise de cobertura, correlação com desfechos, ranking de gaps.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np

from src.analysis import (
    cobertura_por_uf, cobertura_por_regiao, cobertura_por_faixa_etaria,
    cobertura_por_sexo, cobertura_por_raca, correlacao_cobertura_desfecho,
    gap_score, ranking_municipios_criticos, serie_temporal_mensal, resumo_metricas,
)
from src.transform import prepara_pni_completo, agrega_doses

In [ ]:
# Carregar dados processados ou processar novamente
try:
    df = pd.read_parquet("../data/processed/pni_influenza.parquet")
    print(f"Carregado do cache: {len(df):,} registros")
except FileNotFoundError:
    print("Processando dados novamente...")
    import pyarrow.dataset as pds
    import pyarrow.fs as fs
    from src.config import S3_ENDPOINT, S3_ACCESS_KEY, S3_SECRET_KEY, S3_REGION, S3_BUCKET, ANOS, COLUNAS_PNI
    s3 = fs.S3FileSystem(endpoint_override=S3_ENDPOINT, access_key=S3_ACCESS_KEY, secret_key=S3_SECRET_KEY, region=S3_REGION)
    dataset = pds.dataset(S3_BUCKET, filesystem=s3, format="parquet", partitioning="hive")
    table = dataset.to_table(filter=pds.field("ano").isin(ANOS), columns=COLUNAS_PNI)
    df = prepara_pni_completo(table)
    df.to_parquet("../data/processed/pni_influenza.parquet", index=False)

In [ ]:
# Métricas resumidas
doses_agg = agrega_doses(df, "uf_mes").to_pandas()
metricas = resumo_metricas(df, doses_agg)
for k, v in metricas.items():
    print(f"{k}: {v}")

In [ ]:
# Cobertura por UF
cov_uf = cobertura_por_uf(doses_agg)
cov_uf

In [ ]:
# Cobertura por faixa etária
cov_fe = cobertura_por_faixa_etaria(df)
cov_fe

In [ ]:
# Cobertura por sexo
cov_sexo = cobertura_por_sexo(df)
cov_sexo

In [ ]:
# Cobertura por raça
cov_raca = cobertura_por_raca(df)
cov_raca

In [ ]:
# Série temporal mensal
serie = serie_temporal_mensal(df)
serie

In [ ]:
# Gap score (se dados de desfecho estiverem disponíveis)
try:
    internacoes_uf = pd.read_csv("../data/processed/internacoes_agg_uf.csv")
    gaps = gap_score(cov_uf, internacoes_uf)
    gaps[["sg_uf", "cobertura_100k", "gap_score"]].head(10)
except FileNotFoundError:
    print("Dados de desfecho não encontrados. Execute ingestão de SIH/SIM primeiro.")

In [ ]:
# Ranking municípios críticos
ranking = ranking_municipios_criticos(df, top_n=20)
ranking

In [ ]:
# Correlação (se houver dados de desfecho)
try:
    merge_df = pd.read_csv("../data/processed/cobertura_desfecho_merge.csv")
    corr = correlacao_cobertura_desfecho(merge_df, "total_doses", "internacoes")
    print(f"Correlação cobertura × internações: r={corr['r']:.3f}, p={corr['p']:.4f}")
except FileNotFoundError:
    print("Dados merge não encontrados.")

---
**Próximo notebook**: `04_dashboard_proto.ipynb` — protótipo do dashboard